<a href="https://colab.research.google.com/github/A-Peoples/Madden-Generator/blob/main/Madden_Player_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import random
import google.cloud
import seaborn as sns
import math

#**Code (Feel Free to edit)**

In [2]:
ratings24 = pd.read_csv('/content/drive/MyDrive/maddennfl24fullplayerratings.csv')

names = pd.read_csv("/content/drive/MyDrive/Passing_1932_2024.csv")

In [3]:
from pickle import GLOBAL

names.reset_index(drop=True, inplace=True)
names = names.loc[names['P_l_a_y_e_r'] != 'League Average']
names[['First_Name', 'Last_Name', 'null']] = names['P_l_a_y_e_r'].str.split(" ", expand=True)

names = names[['First_Name', 'Last_Name']]

First_Name = names['First_Name'].values
Last_Name = names['Last_Name'].values
Hair_Color = ['Black', 'Blonde', 'Brown', 'Red']
Cut_Height = ['Short', 'Medium', 'Long']

len(names)
def generate_name():
  global gen_name
  cus_choice = input('Do you want a custom name (Y/N): ')
  if (cus_choice == "Y") or (cus_choice == "y"):

    gen_name = input("Enter name: ")
  else:
   gen_name = First_Name[random.randint(-1, len(names))] + " "  + Last_Name[random.randint(-1, 8129)]
  global gen_skincolor
  gen_skincolor = str(random.randint(1, 263))
  print("   Name: " + gen_name)
  print("   Head: " + gen_skincolor)

In [4]:
pd.set_option('expand_frame_repr', False)

In [8]:
def player_check():
  pos = ratings24.Position.unique()
  pos_check = -1
  arc_check = -1
  while pos_check < 0:
    global pos_search
    pos_search = str(input('Positions: '  + str(pos) + '\nInput Position: '))
    filt_rating = ratings24.loc[ratings24['Position'] == pos_search]
    if filt_rating.shape[0] == 0:
      print('Invalid Value')
    elif filt_rating.shape[0] > 0:
      pos_check = 2
  while arc_check < 0:
    global arc_search
    arc = filt_rating['Archetype'].unique()
    arc_search = str(input('Archetypes: ' + str(arc) + "\nInput Archetype: "))
    global filt_arc
    filt_arc = filt_rating.loc[filt_rating['Archetype'] == arc_search]
    if filt_arc.shape[0] == 0:
      print('Invalid Value')
    elif filt_arc.shape[0] > 0:
      arc_check = 2
  global cols
  cols = ['Overall Rating',
        'Speed', 'Acceleration', 'Strength', 'Agility', 'Awareness',
        'Catching', 'Carrying', 'Kick Power',
        'Kick Accuracy', 'Run Block', 'Pass Block', 'Tackle',
        'Break Tackle', 'Jumping', 'Kick Return', 'Injury', 'Stamina',
        'Toughness', 'Trucking', 'Change Of Direction',
        'Ball Carrier Vision', 'Stiff Arm', 'Spin Move', 'Juke Move',
        'Impact Blocking', 'Run Block Power', 'Run Block Finesse',
        'Pass Block Power', 'Pass Block Finesse', 'Lead Block', 'Power Moves',
        'Finesse Moves', 'Block Shedding', 'Pursuit', 'Play Recognition',
        'Man Coverage', 'Zone Coverage', 'Spectacular Catch',
        'Catch In Traffic', 'Short Route Running', 'Medium Route Running',
        'Deep Route Running', 'Hit Power', 'Press', 'Release',
        'Break Sack', 'Throw Under Pressure', 'Throw Power',
        'Throw Accuracy Short', 'Throw Accuracy Mid',
        'Throw Accuracy Deep', 'Play Action', 'Throw On The Run', 'Height',
        'Weight']

  filt_arc = filt_arc[cols]
#cols for filtering


def generate_ratings():
  overall_filt = (input('Input Desired Overall (numbers only/r for random): '))
  if overall_filt == "r":
    overall_filt = random.randint(55, 99)
    print("Generated overall is " + str(overall_filt))
  elif overall_filt.isdigit() == True:
    overall_filt = int(overall_filt)
  overall_check_filt = -1
  check_number = 0
  while overall_check_filt < 0:
    num_min = overall_filt - 3
    num_max = overall_filt + 6

    filt_val = filt_arc.loc[(num_min - check_number < filt_arc['Overall Rating']) & (filt_arc['Overall Rating'] < num_max + check_number)]

    if filt_val.shape[0] > 2:
      print('Overall Filter Range (' + str(filt_val['Overall Rating'].min()) + "-" + str(filt_val['Overall Rating'].max()) + ")\nChecks: " + str(check_number))
      overall_check_filt = 1
    check_number += 1
  #creating final stats
  global final_cols
  final_cols = []
  for col in cols:
    filt_stat = filt_val[col]
    if col == "Awareness":
      if arc_search == 'QB_FieldGeneral':
        added_stat = int(filt_stat.mean() * (random.randint(100, 130) / 100))
    elif col == "Height":
      added_stat = int(filt_stat.mean() * (random.randint(95, 105) / 110))
    else:
      added_stat = int(filt_stat.mean() * (random.randint(85, 110) / 100))
    if (added_stat > 99) & (col != 'Weight'):
      added_stat = 99
    final_cols.append(added_stat)
    #print("Num is " + str(added_stat))
  global final_stats
  final_stats = pd.DataFrame(final_cols, (cols)).transpose()


In [5]:
def print_stats():
  passing_cols = ['Throw Under Pressure', 'Throw Power',
        'Throw Accuracy Short', 'Throw Accuracy Mid',
        'Throw Accuracy Deep', 'Play Action', 'Throw On The Run', 'Break Sack']
  physical_cols = ['Height', 'Weight', 'Speed', 'Acceleration', 'Strength', 'Agility',
        'Awareness', 'Carrying', 'Injury', 'Stamina', 'Toughness']
  rushing_cols = ['Break Tackle', 'Jumping',
        'Trucking', 'Power Moves', 'Finesse Moves',
        'Change Of Direction', 'Ball Carrier Vision', 'Stiff Arm',
        'Spin Move', 'Juke Move']
  catching_cols = ['Catching', 'Spectacular Catch', 'Catch In Traffic', 'Short Route Running',
        'Medium Route Running', 'Deep Route Running', 'Release']
  blocking_cols = ['Run Block', 'Run Block Power','Run Block Finesse',
                  'Pass Block','Pass Block Power', 'Pass Block Finesse','Lead Block', 'Impact Blocking']
  passrush_cols = ['Power Moves', 'Finesse Moves', 'Block Shedding', 'Pursuit',]
  coverage_cols = ['Press', 'Play Recognition', 'Man Coverage', 'Zone Coverage',]
  kicking_cols = [ 'Kick Power', 'Kick Accuracy', 'Kick Return']

  print('   Physical Cols:')
  print(final_stats[physical_cols])

  print('   Passing Cols:')
  print(final_stats[passing_cols])

  print('   Rushing Cols:')
  print(final_stats[rushing_cols])

  print('   Receiving Cols:')
  print(final_stats[catching_cols])

  print('   Blocking Cols:')
  print(final_stats[blocking_cols])

  print('   Coverage Cols:')
  print(final_stats[coverage_cols])

  print('   Pass Rush Cols:')
  print(final_stats[passrush_cols])

  print('   Kicking Cols:')
  print(final_stats[kicking_cols])

#**Generator (Run This)**

In [7]:
player_check()
generate_ratings()
generate_name()
print_stats()

Positions: ['C' 'CB' 'DT' 'FB' 'FS' 'HB' 'K' 'LE' 'LG' 'LOLB' 'LT' 'MLB' 'P' 'QB'
 'RE' 'RG' 'ROLB' 'RT' 'SS' 'TE' 'WR']
Input Position: HB
Archetypes: ['HB_ReceivingBack' 'HB_ElusiveBack' 'HB_PowerBack']
Input Archetype: HB_PowerBack
Input Desired Overall (numbers only/r for random): 82
Overall Filter Range (80-85)
Checks: 0
Do you want a custom name (Y/N): N
   Name: Rodney Pitts
   Head: 179
   Physical Cols:
   Height  Weight  Speed  Acceleration  Strength  Agility  Awareness  Carrying  Injury  Stamina  Toughness
0      77     222     93            91        87       73         73        89      84       84         92
   Passing Cols:
   Throw Under Pressure  Throw Power  Throw Accuracy Short  Throw Accuracy Mid  Throw Accuracy Deep  Play Action  Throw On The Run  Break Sack
0                    15           31                    22                  20                   15           16                23          28
   Rushing Cols:
   Break Tackle  Jumping  Trucking  Power Moves  F

In [14]:
final_stats['Name'] = gen_name
final_stats['Pos'] = pos_search
final_stats['Head'] = gen_skincolor
final_stats

,Overall Rating,Speed,Acceleration,Strength,Agility,Awareness,Catching,Carrying,Kick Power,Kick Accuracy,...,Throw Accuracy Short,Throw Accuracy Mid,Throw Accuracy Deep,Play Action,Throw On The Run,Height,Weight,Name,Pos,Head
0,73,93,91,87,73,73,61,89,16,14,...,22,20,15,16,23,77,222,Rodney Pitts,HB,179
